# Lab 4 — ROC Curve & AUC

**Day 03 · Classification & Model Interpretation · Cisco AI/ML Training**

---

## Learning objectives

1. Build an **ROC curve** from predicted probabilities at many thresholds.
2. Compute **AUC** (area under the ROC curve).
3. Compare model performance to a **random classifier** (diagonal line).
4. Relate ROC to the threshold trade-offs from Lab 3.

> **Checkpoints:** AUC ≈ **0.63** · `roc_curve.png` saved · AUC **> 0.5**



## Why ROC when we already have precision/recall?

Lab 3 used a **single threshold** (0.5). ROC shows performance at **all** thresholds in one chart:

| Axis | Meaning |
|------|---------|
| **X — FPR** | False positive rate = FP / (FP + TN) |
| **Y — TPR** | True positive rate (recall) = TP / (TP + FN) |

**AUC** summarizes ranking quality:
- **0.5** = random guessing
- **1.0** = perfect separation
- **0.63** = modest but useful lift over chance


---

## 1. Train model and get probability scores


In [ ]:
%matplotlib inline

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import Image, display
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import auc, roc_auc_score, roc_curve
from sklearn.model_selection import train_test_split

GH_ROOT = Path.cwd().resolve()
if GH_ROOT.name == "notebooks":
    GH_ROOT = GH_ROOT.parents[2]
elif GH_ROOT.name == "day-03":
    GH_ROOT = GH_ROOT.parents[1]
else:
    for parent in [GH_ROOT, *GH_ROOT.parents]:
        if (parent / "data" / "lending-club" / "lending_club_sample.csv").is_file():
            GH_ROOT = parent
            break

OUTPUT_DIR = GH_ROOT / "hands-on" / "day-03" / "output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DEFAULT_STATUSES = {"Charged Off", "Late (31-120 days)"}
df = pd.read_csv(GH_ROOT / "data" / "lending-club" / "lending_club_sample.csv")
df["default"] = df["loan_status"].isin(DEFAULT_STATUSES).astype(int)

NUMERIC_FEATURES = ["loan_amnt", "int_rate", "annual_inc", "dti", "installment"]
X, y = df[NUMERIC_FEATURES], df["default"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

model = LogisticRegression(max_iter=1000, random_state=42)
model.fit(X_train, y_train)
y_scores = model.predict_proba(X_test)[:, 1]

print(f"test scores: min={y_scores.min():.3f}, max={y_scores.max():.3f}")


`y_scores` = P(default=1) — the **ranking** signal ROC evaluates. We do not fix a threshold yet.


---

## 2. Compute ROC curve points


In [ ]:
fpr, tpr, thresholds = roc_curve(y_test, y_scores)
roc_auc = auc(fpr, tpr)

print("Lab 4 — ROC and AUC")
print(f"ROC points: {len(fpr)}")
print(f"AUC (trapezoid): {roc_auc:.4f}")
print(f"AUC (sklearn shortcut): {roc_auc_score(y_test, y_scores):.4f}")
print(f"threshold at index 10: {thresholds[10]:.4f}")


Each point on the curve = one classification threshold. Moving left→right on the curve = lowering the threshold (more positives flagged).


---

## 3. Plot ROC vs random baseline


In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
ax.plot(fpr, tpr, lw=2, label=f"Logistic model (AUC = {roc_auc:.3f})")
ax.plot([0, 1], [0, 1], linestyle="--", color="gray", label="Random (AUC = 0.5)")
ax.set_xlabel("False positive rate")
ax.set_ylabel("True positive rate (recall)")
ax.set_title("ROC — loan default model")
ax.legend(loc="lower right")
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
plt.tight_layout()

roc_plot = OUTPUT_DIR / "roc_curve.png"
fig.savefig(roc_plot, dpi=100, bbox_inches="tight")
plt.show()
print(f"plot saved: {roc_plot.name}")
assert roc_plot.is_file()


In [ ]:
display(Image(filename=str(roc_plot)))


**Diagonal dashed line:** A random classifier that scores loans randomly — AUC = 0.5. Our curve **above** the diagonal → model adds value.


---

## 4. Optimal threshold by Youden's J (extension)

**Youden's J** = TPR − FPR. Maximizing J picks a threshold balancing sensitivity and specificity.


In [ ]:
youden_j = tpr - fpr
best_idx = int(np.argmax(youden_j))
best_threshold = thresholds[best_idx]

print(f"Best Youden index at idx {best_idx}")
print(f"  threshold: {best_threshold:.4f}")
print(f"  TPR: {tpr[best_idx]:.4f}, FPR: {fpr[best_idx]:.4f}")
print(f"  J = {youden_j[best_idx]:.4f}")

# Mark on plot
fig, ax = plt.subplots(figsize=(6, 5))
ax.plot(fpr, tpr, lw=2)
ax.scatter(fpr[best_idx], tpr[best_idx], color="red", s=80, zorder=5, label=f"max J, t={best_threshold:.2f}")
ax.plot([0, 1], [0, 1], linestyle="--", color="gray")
ax.set_xlabel("FPR"); ax.set_ylabel("TPR")
ax.set_title("ROC with best Youden point")
ax.legend()
plt.tight_layout()
plt.show()


---

## 5. AUC interpretation for credit risk

| AUC range | Typical interpretation |
|-----------|------------------------|
| 0.5 | No discrimination |
| 0.6 – 0.7 | Acceptable for pilot models |
| 0.7 – 0.8 | Good |
| 0.8+ | Strong (rare in messy real data) |

Our **~0.63** is reasonable for a quick logistic model on five numeric features — feature engineering and pipelines (Lab 5) may help slightly.


---

## 6. Checkpoint summary


In [ ]:
assert roc_auc > 0.5
assert abs(roc_auc - 0.6326) < 0.02
assert roc_plot.is_file()
print("✓ All checkpoint assertions passed")


---

## Reflection questions

1. Can AUC be high while precision at 0.5 threshold is mediocre? Why?
2. When would you prefer optimizing recall over AUC?
3. How is ROC related to the threshold table in Lab 3?

**Previous:** [Lab 3 — Confusion matrix](lab03_confusion_matrix.ipynb)  
**Next:** [Lab 5 — sklearn Pipeline](lab05_sklearn_pipeline.ipynb)
